In [34]:
import os
import pandas as pd

In [35]:
from typing import Dict, Tuple
import numpy as np
import pandas as pd

def _to_bin(x) -> np.ndarray:
    """Coerce a series/array to {0,1} int array."""
    s = pd.Series(x).fillna(0)
    s = s.replace({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    s = pd.to_numeric(s, errors='coerce').fillna(0.0)
    return (s > 0.5).astype(int).to_numpy()

def _f1_prec_rec(y_true: np.ndarray, y_pred: np.ndarray):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1, precision, recall


def optimize_pds_threshold(
    gt_df: pd.DataFrame,
    pds_df: pd.DataFrame,
    pedal_mapping: Dict[str, str],
    raw_pressure_mapping: Dict[str, str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    For each pedal in pedal_mapping:
      - search a scalar threshold on the corresponding raw pressure column to maximize F1 vs GT
      - overwrite pds_df[pds_col] with the new binary based on the best threshold/polarity
    Returns (updated_pds_df, summary_df)
    """
    # trim to common length to keep indexing simple
    n = min(len(gt_df), len(pds_df))
    gt_df = gt_df.iloc[:n].reset_index(drop=True)
    pds_df = pds_df.iloc[:n].reset_index(drop=True)

    summaries = []

    for gt_col, pds_col in pedal_mapping.items():
        print("\n*-----------------------------*")
        print(f"Optimizing threshold for {gt_col} → PDS column '{pds_col}'")

        # checks
        if gt_col not in gt_df.columns:
            print(f"❌ Missing GT column: {gt_col}")
            continue
        if pds_col not in pds_df.columns:
            print(f"❌ Missing PDS binary column: {pds_col}")
            continue
        if gt_df[gt_col].sum() == 0:
            print(f"⚠️  Skipping {gt_col}: no positives in GT (nothing to optimize against).")
            continue

        press_col = raw_pressure_mapping.get(gt_col)
        if not press_col or press_col not in pds_df.columns:
            print(f"❌ Missing pressure column for {gt_col}: '{press_col}'")
            continue

        # search best threshold (no lag)
        best = determine_best_threshold_simple(
            gt_series=gt_df[gt_col],
            pressure_series=pds_df[press_col],
            n_thresh=200
        )
        if not best.get("success", False):
            print(f"❌ Optimization failed for {gt_col}: {best.get('reason')}")
            continue

        thr = best["threshold"]
        pol = best["polarity"]

        # ---- APPLY to pds_df (overwrite Pedal X Pressed) ----
        raw = pd.to_numeric(pds_df[press_col], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy()
        pred = (raw >= thr).astype(int) if pol == ">=" else (raw <= thr).astype(int)
        pds_df[pds_col] = pred  # overwrite in-place

        # sanity metrics (recompute vs GT after applying)
        y_true = (pd.to_numeric(gt_df[gt_col], errors="coerce").fillna(0) > 0.5).astype(int).to_numpy()
        f1, prec, rec = _f1_prec_rec(y_true, pred)

        print(f"✅ Best threshold for {gt_col}: {thr:.6f} {pol} | F1={f1:.4f} P={prec:.4f} R={rec:.4f}")

        summaries.append(dict(
            Pedal=gt_col,
            PDS_Column=pds_col,
            Pressure_Column=press_col,
            Threshold=float(thr),
            Polarity=pol,
            F1=float(f1),
            Precision=float(prec),
            Recall=float(rec),
            Positives_GT=int(y_true.sum()),
            Frames=int(len(y_true)),
        ))

    summary_df = pd.DataFrame(summaries)
    return pds_df, summary_df


In [40]:

# ---------------- config ----------------
ROOT_PATH = "/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/"
TRIALS = [
    "S105_T1",
    "S106_T1","S106_T2","S112_T1","S112_T2","S116_T1","S116_T2","S116_T4","S116_T5",
    "S118_T1","S200_T1","S201_T1","S201_T2","S202_T1","S203_T1","S204_T1","S209_T2","S210_T1",
    "S214_T1","S214_T4","S214_T6","S215_T3","S215_T4","S217_T2","S217_T3","S217_T4","S218_T1","S219_T1",
]

# Your mapping between GT columns and PDS sync columns
pedal_mapping = {
    'Upper Left':   'Pedal 1 Pressed',
    'Upper Right':  'Pedal 2 Pressed',
    'Lower Left':   'Pedal 3 Pressed',
    'Lower Right':  'Pedal 4 Pressed',
    'Camera_Pedal': 'Pedal 6 Pressed',
    'Arm_Swap':     'Pedal 7 Pressed'
}

raw_pressure_mapping = {
    'Upper Left':   'Pedal 1 Pressure',
    'Upper Right':  'Pedal 2 Pressure',
    'Lower Left':   'Pedal 3 Pressure',
    'Lower Right':  'Pedal 4 Pressure',
    'Camera_Pedal': 'Pedal 6 Pressure',
    'Arm_Swap':     'Pedal 7 Pressure'
}


for trial in TRIALS:
    gt_path  = f"{ROOT_PATH}/{trial}/synched_data/{trial}_pds_gt.csv"
    pds_path = f"{ROOT_PATH}/{trial}/synched_data/final_annotation_{trial}.csv"

    if not os.path.isfile(gt_path):
        print(f"❌ Missing GT: {gt_path} — skipping")
        continue
    if not os.path.isfile(pds_path):
        print(f"❌ Missing PDS sync: {pds_path} — skipping")
        continue

    print(f"\n=== Trial: {trial} ===")

    gt_df   = pd.read_csv(gt_path)
    pds_df  = pd.read_csv(pds_path)


    pds_df_optimized, summary_df = optimize_pds_threshold(
        gt_df=gt_df,
        pds_df=pds_df,
        pedal_mapping=pedal_mapping,
        raw_pressure_mapping=raw_pressure_mapping
    )

    print("\nOptimization complete:")

    # print(pds_df_optimized.head())
    # save the otimized pds
    optimized_path = f"{ROOT_PATH}/{trial}/synched_data/final_annotation_{trial}_optimized.csv"
    pds_df_optimized.to_csv(optimized_path, index=False)
    print(f"Optimized PDS data saved to: {optimized_path}")


=== Trial: S105_T1 ===


/tmp/ipykernel_472550/3162143545.py:44: DtypeWarning: Columns (66,67,68,69,70) have mixed types. Specify dtype option on import or set low_memory=False.
  pds_df  = pd.read_csv(pds_path)
/tmp/ipykernel_472550/773584261.py:78: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  p  = pd.to_numeric(pd.Series(pressure_series), errors='coerce').fillna(method='ffill').fillna(method='bfill').to_numpy()



*-----------------------------*
Optimizing threshold for Upper Left → PDS column 'Pedal 1 Pressed'
⚠️  Skipping Upper Left: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Upper Right → PDS column 'Pedal 2 Pressed'
⚠️  Skipping Upper Right: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Lower Left → PDS column 'Pedal 3 Pressed'
⚠️  Skipping Lower Left: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Lower Right → PDS column 'Pedal 4 Pressed'
✅ Best threshold for Lower Right: 10537.370854 >= | F1=0.6190 P=0.5714 R=0.6753

*-----------------------------*
Optimizing threshold for Camera_Pedal → PDS column 'Pedal 6 Pressed'


/tmp/ipykernel_472550/2480362246.py:75: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  raw = pd.to_numeric(pds_df[press_col], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy()
/tmp/ipykernel_472550/773584261.py:78: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  p  = pd.to_numeric(pd.Series(pressure_series), errors='coerce').fillna(method='ffill').fillna(method='bfill').to_numpy()
/tmp/ipykernel_472550/2480362246.py:75: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  raw = pd.to_numeric(pds_df[press_col], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy()
/tmp/ipykernel_472550/773584261.py:78: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future versio

✅ Best threshold for Camera_Pedal: 1450.065327 >= | F1=0.5772 P=0.4528 R=0.7958

*-----------------------------*
Optimizing threshold for Arm_Swap → PDS column 'Pedal 7 Pressed'
✅ Best threshold for Arm_Swap: 13.763819 >= | F1=0.0516 P=0.0266 R=0.8425

Optimization complete:


...

✅ Best threshold for Camera_Pedal: 350.060302 >= | F1=0.6680 P=0.6812 R=0.6553

*-----------------------------*
Optimizing threshold for Arm_Swap → PDS column 'Pedal 7 Pressed'
✅ Best threshold for Arm_Swap: 9.085427 <= | F1=0.0412 P=0.0210 R=1.0000

Optimization complete:


/tmp/ipykernel_472550/2480362246.py:75: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  raw = pd.to_numeric(pds_df[press_col], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy()


Optimized PDS data saved to: /standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed//S218_T1/synched_data/final_annotation_S218_T1_optimized.csv

=== Trial: S219_T1 ===


/tmp/ipykernel_472550/3162143545.py:44: DtypeWarning: Columns (62,63,64,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  pds_df  = pd.read_csv(pds_path)
/tmp/ipykernel_472550/773584261.py:78: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  p  = pd.to_numeric(pd.Series(pressure_series), errors='coerce').fillna(method='ffill').fillna(method='bfill').to_numpy()
/tmp/ipykernel_472550/2480362246.py:75: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  raw = pd.to_numeric(pds_df[press_col], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy()
/tmp/ipykernel_472550/773584261.py:78: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  p  = pd.to_numeric(pd.Series(pressure_series), errors='c


*-----------------------------*
Optimizing threshold for Upper Left → PDS column 'Pedal 1 Pressed'
⚠️  Skipping Upper Left: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Upper Right → PDS column 'Pedal 2 Pressed'
⚠️  Skipping Upper Right: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Lower Left → PDS column 'Pedal 3 Pressed'
⚠️  Skipping Lower Left: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Lower Right → PDS column 'Pedal 4 Pressed'
⚠️  Skipping Lower Right: no positives in GT (nothing to optimize against).

*-----------------------------*
Optimizing threshold for Camera_Pedal → PDS column 'Pedal 6 Pressed'
✅ Best threshold for Camera_Pedal: 1192.494975 >= | F1=0.4201 P=0.3248 R=0.5944

*-----------------------------*
Optimizing threshold for Arm_Swap → PDS column 'Pedal 7 Pressed'
✅ Best threshold